# GB1 optimization with cached + batched ESM embeddings

This is the stock `protein_optimization_GB1.ipynb` with three changes:

| stock | here |
|---|---|
| `ProteinEmbedding` | `CachedProteinEmbedding` (memoizes per sequence, caps forward-pass size) |
| `GPLLModel` | `CachedGPLLModel` (pre-warms the cache in bounded batches) |
| - | `plm.add_sequences(..., pin=True)` before each round |

## Why the GA was still using so much memory

`gpytorch.models.ExactGP.__call__` does something non-obvious in eval mode: it
**concatenates the stored training inputs with the test inputs** and evaluates
`forward()` on the joint tensor:

```python
# gpytorch/models/exact_gp.py, _get_test_prior_mean_and_covariances
full_inputs.append(torch.cat([train_input, input], dim=-2))
full_output = super().__call__(*full_inputs, **kwargs)
```

`_ExactGPLLModel.forward()` immediately calls `self.transformer.embed(tokens)`
on whatever it is handed. So every `GPLLModel.predict()` reaches ESM with
**(n_train + n_test)** sequences, not the n_test trial sequences you would
expect.

SequenceGA calls `predict()` once per generation, through
`Problem._evaluate` -> `acq_fun.forward` -> `surrogate_model.predict`. With the
defaults here (`n_pop=500`, `period=15`) that is dozens of calls per
`recommand()`, each presenting the entire accumulated training pool plus the
whole population to ESM in one tensor. By round 5 the training pool is 576
sequences, so each call is a single forward pass over 1,076 sequences.

Caching alone fixes the redundant *compute* but not the *peak*: the misses
still have to be embedded, and v1 of the cache embedded them all in one go.
`embed_batch_size` is what bounds the peak.

Measured on a simulated 10-generation run (96 training sequences, `n_pop=500`):

| | largest single ESM batch | sequences through ESM |
|---|---|---|
| stock | **596** | 6,056 |
| cached + batched (`embed_batch_size=64`) | **64** | 950 |

Predictions are numerically identical (max abs difference 0.0).


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from gpytorch.kernels import RBFKernel
from gpytorch.priors import NormalPrior

from mobius import Planner, SequenceGA
from mobius import CachedProteinEmbedding          # <-- was ProteinEmbedding
from mobius import InverseFolding
from mobius import CachedGPLLModel                 # <-- was GPLLModel
from mobius import ExpectedImprovement
from mobius.utils import generate_biopolymer_design_protocol_from_probabilities
from mobius import homolog_scanning
from mobius import convert_FASTA_to_HELM, convert_HELM_to_FASTA


## GB1 dataset

Citation: Adaptation in protein fitness landscapes is facilitated by indirect paths; Wu et al.; 2016; http://dx.doi.org/10.7554/eLife.16965.001

In [ ]:
df_exp = pd.read_excel('elife-16965-supp1-v4.xlsx')
df_imputated = pd.read_excel('elife-16965-supp2-v4.xlsx')

In [ ]:
df_imputated = df_imputated.rename(columns={"Imputed fitness": "Fitness"})
df = pd.concat([df_exp[['Variants', 'Fitness']], df_imputated])

In [ ]:
sequences = []
wt_sequence = list('MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE')

for variant in df['Variants'].values:
    wt_sequence[38] = variant[0]
    wt_sequence[39] = variant[1]
    wt_sequence[40] = variant[2]
    wt_sequence[53] = variant[3]

    sequences.append(''.join(wt_sequence))

df['Sequences'] = sequences
df.to_csv('GB1_dataset.csv', index=False)

## Initialize GB1 oracle

In [ ]:
class GB1Scorer:
    
    def __init__(self, sequences, values):
        if len(sequences) != len(values):
            raise ValueError(f'Different numbers of sequences and values ({len(sequences)} != {len(values)})')

        self._data = {s: v for s, v in zip(sequences, values)}

    def score(self, sequences):
        if not isinstance(sequences, (list, tuple, np.ndarray)):
            sequences = [sequences]
        
        return np.array([self._data[s] for s in sequences])


In [ ]:
df = pd.read_csv('GB1_dataset.csv')
sequences = df['Sequences'].values
fitness = df['Fitness'].values

gb1 = GB1Scorer(sequences, fitness)
gb1.score('MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE')

## Some plots...

In [ ]:
print(np.min(fitness))
print(np.max(fitness))

In [ ]:
fig, ax = plt.subplots(figsize=(20, 7.5))
ax.hist(fitness, bins=np.linspace(0, 10, 101), log=True)
#ax.set_ylim([-0.1, 500])

plt.show()

## Run Bayesian Optimization

### With ESM-1b only

In [ ]:
# Load protein language model.
#
# embed_batch_size is the memory knob: it caps how many sequences are ever
# sent through ESM in a single forward pass. Nothing else about the run
# changes when you adjust it - only peak VRAM and throughput.
#   * OOM on a small card?  drop it to 16 or 32
#   * lots of headroom?     raise it to 128+
#
# max_cache_size bounds *host* RAM. This GB1 design varies 4 positions, so the
# reachable space is 20^4 = 160,000 sequences; at 1280 floats each that is
# ~820 MB if the GA explores widely. 50,000 keeps it under ~250 MB. Set it to
# None for an unbounded cache.
plm = CachedProteinEmbedding(pretrained_model_name='esm1b_t33_650M_UR50S',
                             embedding_type='avg',
                             embed_batch_size=64,
                             max_cache_size=50_000)


In [ ]:
yaml_content = """
    design:
      monomers:
        default: [A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y]
      biopolymers:
        - name: GB1
          starting_residue: 1
          length: 56
          positions:
            39: default
            40: default
            41: default
            54: default
    """
    
with open('design_protocol_gb1.yaml', 'w') as f:
    f.write(yaml_content)

In [ ]:
# Load surrogate model.
#
# CachedGPLLModel is a drop-in for GPLLModel: it pre-warms the embedding cache
# in bounded mini-batches before handing anything to gpytorch, and pins the
# training set so LRU can never evict it. Everything downstream - the
# acquisition function, SequenceGA, Planner - is untouched.
gpmodel = CachedGPLLModel(kernel=RBFKernel(), pretrained_model=plm, noise_prior=NormalPrior(0, 1))
ei = ExpectedImprovement(gpmodel, maximize=True)
optimizer = SequenceGA(algorithm='GA', period=15, design_protocol_filename='design_protocol_gb1.yaml')
ps = Planner(ei, optimizer)


In [ ]:
lead_protein = convert_FASTA_to_HELM('MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE')[0]

seed_library = [lead_protein]
positions = {'PEPTIDE1': [39, 40, 41, 54]}

for seq in homolog_scanning(lead_protein, positions=positions):
    seed_library.append(seq)

    if len(seed_library) >= 96:
        print('Reach max. number of protein allowed.')
        break

seed_library = convert_HELM_to_FASTA(seed_library)
scores_seed_library = gb1.score(seed_library)

# Pre-warm the cache with the seed library before the first round, in bounded
# batches, and pin it: gpytorch re-reads every training row on every single
# predict() call, so these must never be evicted.
plm.add_sequences(seed_library, pin=True)
print(f'Cache pre-warmed: {plm.cache_info()}')


In [ ]:
sequences = seed_library.copy()
scores = scores_seed_library.copy()

# Store data for later analysis
data = [(0, p, s) for p, s in zip(sequences, scores)]

for i in range(5):
    suggested_sequences, _ = ps.recommand(sequences, scores.reshape(-1, 1), batch_size=96)
    scores_suggested_sequences = gb1.score(suggested_sequences)

    sequences = np.concatenate([sequences, suggested_sequences])
    scores = np.concatenate((scores, scores_suggested_sequences), axis=0)
    data.extend([(i + 1, p, s) for p, s in zip(suggested_sequences, scores_suggested_sequences)])

    # The pool that will become the GP's training set next round. Pre-warming
    # and pinning it here means the next round's fit() and every one of its
    # predict() calls read these rows straight from cache.
    plm.add_sequences(sequences, pin=True)

    best_seq = sequences[np.argmax(scores)]
    best_score = np.max(scores)
    min_score = np.min(scores_suggested_sequences)
    mean_score = np.mean(scores_suggested_sequences)
    print(f'Best GB1 found so far: {best_seq} / {best_score:.3f} (min: {min_score:.3f}, mean: {mean_score:.3f}, max: {best_score:.3f})')

    info = plm.cache_info()
    print(f"  cache: {info['n_embedded']:,} unique embedded | "
          f"hit rate {info['hit_rate']:.1%} | "
          f"{info['forward_passes']:,} ESM forward passes | "
          f"largest ESM batch {info['max_forward_batch']} (cap {info['embed_batch_size']})")
    print('')

    df = pd.DataFrame(data=data, columns=('iter', 'sequence', 'exp_value'))
    df.to_csv('results_gb1_homolog.csv', index=False)


### With ESM-1b and ESM-IF1 (T=0.5)

In [ ]:
# Load protein language model.
#
# embed_batch_size is the memory knob: it caps how many sequences are ever
# sent through ESM in a single forward pass. Nothing else about the run
# changes when you adjust it - only peak VRAM and throughput.
#   * OOM on a small card?  drop it to 16 or 32
#   * lots of headroom?     raise it to 128+
#
# max_cache_size bounds *host* RAM. This GB1 design varies 4 positions, so the
# reachable space is 20^4 = 160,000 sequences; at 1280 floats each that is
# ~820 MB if the GA explores widely. 50,000 keeps it under ~250 MB. Set it to
# None for an unbounded cache.
plm = CachedProteinEmbedding(pretrained_model_name='esm1b_t33_650M_UR50S',
                             embedding_type='avg',
                             embed_batch_size=64,
                             max_cache_size=50_000)


In [ ]:
# Get probabilities from structure model
iv = InverseFolding()
probabilities = iv.get_probabilities_from_structure('2gi9.pdb', target_chainid='A', temperature=0.5)
monomers = iv.vocab

# Fix positions involved in the fluorophore activity
wt_seq = 'MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE'
positions = [39, 40, 41, 54]
fixed_positions = {}

for i, aa in enumerate(wt_seq):
    if not i + 1 in positions:
        fixed_positions[i + 1] = aa

with open('design_protocol_gb1_probabilities.yaml', 'w') as w:
    data = generate_biopolymer_design_protocol_from_probabilities(probabilities, monomers, fixed_positions=fixed_positions)
    yaml.dump(data, w, sort_keys=False)

In [ ]:
# Load surrogate model.
#
# CachedGPLLModel is a drop-in for GPLLModel: it pre-warms the embedding cache
# in bounded mini-batches before handing anything to gpytorch, and pins the
# training set so LRU can never evict it. Everything downstream - the
# acquisition function, SequenceGA, Planner - is untouched.
gpmodel = CachedGPLLModel(kernel=RBFKernel(), pretrained_model=plm, noise_prior=NormalPrior(0, 1))
ei = ExpectedImprovement(gpmodel, maximize=True)
optimizer = SequenceGA(algorithm='GA', period=15, design_protocol_filename='design_protocol_gb1_probabilities.yaml')
ps = Planner(ei, optimizer)


In [ ]:
lead_protein = convert_FASTA_to_HELM('MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE')[0]

seed_library = [lead_protein]
positions = {'PEPTIDE1': [39, 40, 41, 54]}

for seq in homolog_scanning(lead_protein, positions=positions):
    seed_library.append(seq)

    if len(seed_library) >= 96:
        print('Reach max. number of protein allowed.')
        break

seed_library = convert_HELM_to_FASTA(seed_library)
scores_seed_library = gb1.score(seed_library)

# Pre-warm the cache with the seed library before the first round, in bounded
# batches, and pin it: gpytorch re-reads every training row on every single
# predict() call, so these must never be evicted.
plm.add_sequences(seed_library, pin=True)
print(f'Cache pre-warmed: {plm.cache_info()}')


In [ ]:
sequences = seed_library.copy()
scores = scores_seed_library.copy()

# Store data for later analysis
data = [(0, p, s) for p, s in zip(sequences, scores)]

for i in range(5):
    suggested_sequences, _ = ps.recommand(sequences, scores.reshape(-1, 1), batch_size=96)
    scores_suggested_sequences = gb1.score(suggested_sequences)

    sequences = np.concatenate([sequences, suggested_sequences])
    scores = np.concatenate((scores, scores_suggested_sequences), axis=0)
    data.extend([(i + 1, p, s) for p, s in zip(suggested_sequences, scores_suggested_sequences)])

    # The pool that will become the GP's training set next round. Pre-warming
    # and pinning it here means the next round's fit() and every one of its
    # predict() calls read these rows straight from cache.
    plm.add_sequences(sequences, pin=True)

    best_seq = sequences[np.argmax(scores)]
    best_score = np.max(scores)
    min_score = np.min(scores_suggested_sequences)
    mean_score = np.mean(scores_suggested_sequences)
    print(f'Best GB1 found so far: {best_seq} / {best_score:.3f} (min: {min_score:.3f}, mean: {mean_score:.3f}, max: {best_score:.3f})')

    info = plm.cache_info()
    print(f"  cache: {info['n_embedded']:,} unique embedded | "
          f"hit rate {info['hit_rate']:.1%} | "
          f"{info['forward_passes']:,} ESM forward passes | "
          f"largest ESM batch {info['max_forward_batch']} (cap {info['embed_batch_size']})")
    print('')

    df = pd.DataFrame(data=data, columns=('iter', 'sequence', 'exp_value'))
    df.to_csv('results_gb1_homolog_esm-if1.csv', index=False)


## Comparison

In [ ]:
methods = ['gb1_homolog', 'gb1_homolog_esm-if1']
method_renaming = {'gb1_homolog': 'Homolog + ESM-1b', 'gb1_homolog_esm-if1': 'Homolog + ESM-1b + ESM-IF1 (T=0.5)'}

dfs = []

for method in methods:
    df = pd.read_csv(f'results_{method}.csv')
    df['method'] = method_renaming[method]
    maxs = df.loc[df.groupby(by=['iter'])['exp_value'].idxmax()]
    dfs.append(maxs)

dfs = pd.concat(dfs).reset_index()

dfs['iter'] = dfs['iter'].replace({0: 'Init.', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5'})

fig, axarr = plt.subplots(figsize=(10, 5))
sns.lineplot(x='iter', y='exp_value', hue='method', data=dfs, ax=axarr, linewidth=3.5)

axarr.legend(loc='upper right', fontsize=15, frameon=False)
axarr.set_ylim([-0.1, 10.1])
axarr.set_xlabel("Generations", fontsize=15)
axarr.set_ylabel("GB1 Binding Fitness", fontsize=15)
axarr.xaxis.set_tick_params(labelsize=15)
axarr.yaxis.set_tick_params(labelsize=15)

axarr.get_legend().remove()
axarr.legend(bbox_to_anchor=(0., 1., 1., 0.2), loc=3, ncol=2, mode="expand", borderaxespad=0, fontsize=15, frameon=False)

sns.despine()

plt.show()

In [ ]:
methods = ['gb1_homolog', 'gb1_homolog_esm-if1']
method_renaming = {'gb1_homolog': 'Homolog + ESM-1b', 'gb1_homolog_esm-if1': 'Homolog + ESM-1b + ESM-IF1 (T=0.5)'}

dfs = []

for method in methods:
    df = pd.read_csv(f'results_{method}.csv')
    df['method'] = method_renaming[method]
    dfs.append(df)

dfs = pd.concat(dfs).reset_index()

dfs['iter'] = dfs['iter'].replace({0: 'Init.', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5'})

fig, axarr = plt.subplots(figsize=(10, 5))
sns.lineplot(x='iter', y='exp_value', hue='method', data=dfs, ax=axarr, linewidth=3.5)

axarr.legend(loc='upper right', fontsize=15, frameon=False)
axarr.set_ylim([-0.1, 10.1])
axarr.set_xlabel("Generations", fontsize=15)
axarr.set_ylabel("GB1 Binding Fitness", fontsize=15)
axarr.xaxis.set_tick_params(labelsize=15)
axarr.yaxis.set_tick_params(labelsize=15)

axarr.get_legend().remove()
axarr.legend(bbox_to_anchor=(0., 1., 1., 0.2), loc=3, ncol=2, mode="expand", borderaxespad=0, fontsize=15, frameon=False)

sns.despine()

plt.show()

## Cache summary


In [ ]:
info = plm.cache_info()
for k, v in info.items():
    print(f'{k:>20}: {v}')

assert info['max_forward_batch'] <= info['embed_batch_size'], \
    'a forward pass exceeded the configured cap'
print('\nLargest ESM batch stayed within the configured cap.')
